# NB_patient_360
**Gold Table:** `caresync_databricks.gold.patient_360`  
**Grain:** One row per patient  
**Driving Table:** `caresync_databricks.silver.patients`  
**Load Strategy:** Full rebuild — `CREATE OR REPLACE TABLE ... AS SELECT`  
**Joins:**
- LEFT JOIN `hospitals` on `registered_hospital_id = hospital_id`
- LEFT JOIN `insurance_providers` on `insurance_provider_id`
- LEFT JOIN aggregated `lab_results` (grouped by `patient_id`)

**Source:** STTM — Silver to Gold (CareSync Health Network) v1.0

In [0]:
dbutils.widgets.text("catalog", "caresync_databricks", "Catalog Name")
catalog = dbutils.widgets.get("catalog")
print(f"Using catalog: {catalog}")

In [0]:
%sql
CREATE OR REPLACE TABLE caresync_databricks.gold.patient_360 AS

WITH lab_agg AS (
    SELECT
        patient_id,
        COUNT(*)                                                        AS total_tests,
        COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)         AS abnormal_test_count,
        COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)         AS critical_test_count,
        MAX(test_date)                                                  AS last_test_date
    FROM caresync_databricks.silver.lab_results
    GROUP BY patient_id
)

SELECT
    p.patient_id,
    p.first_name,
    p.last_name,
    CAST(p.date_of_birth AS DATE)                                       AS date_of_birth,
    FLOOR(DATEDIFF(CURRENT_DATE(), CAST(p.date_of_birth AS DATE)) / 365.25)  AS age,
    p.gender,
    p.city,
    p.state,
    p.registered_hospital_id                                            AS hospital_id,
    h.hospital_name,
    p.insurance_provider_id,
    ip.provider_name                                                    AS insurance_provider_name,
    COALESCE(la.total_tests, 0)                                         AS total_tests,
    COALESCE(la.abnormal_test_count, 0)                                 AS abnormal_test_count,
    COALESCE(la.critical_test_count, 0)                                 AS critical_test_count,
    la.last_test_date

FROM caresync_databricks.silver.patients p

LEFT JOIN caresync_databricks.silver.hospitals h
    ON p.registered_hospital_id = h.hospital_id

LEFT JOIN caresync_databricks.silver.insurance_providers ip
    ON p.insurance_provider_id = ip.insurance_provider_id

LEFT JOIN lab_agg la
    ON p.patient_id = la.patient_id

In [0]:
print("[STEP 4] Computing processed record count")
record_count = spark.table(f"caresync_databricks.gold.patient_360").count()
print(f"[STEP 4 COMPLETE] Processed record count: {record_count}")
dbutils.notebook.exit(str(record_count))